# NDT7 (M-Lab) Data Prep — Laos Broadband + Mobile, Province x Quarter

Aggregates `../../../data/ndt7/la/mlab_la_clean.parquet` into province x quarter format, split into
Broadband and Mobile/Cellular parts, mirroring the same structure across all NDT7 "tigger"
countries (Cambodia/Thailand/Vietnam/Laos).

**Ported from Laos's own original notebook** (`notebooks/ndt7/lao/la_clean/ndt7_Laos_eda.ipynb`),
which queried the raw parquet directly per-section via DuckDB (no tile-binning, no separate
province x quarter export) — this prep notebook is new engineering, not a straight port: it applies
the same generic zoom-16 tile-binning + weighted-aggregation SQL used by
`cambodia_ndt7_prep.ipynb` / `thailand_ndt7_prep.ipynb` / `vietnam_ndt7_prep.ipynb` to Laos's raw
data, so `n_tiles` / `is_reliable` stay comparable across every NDT7 country and Ookla.

**Province name mapping** — raw parquet 'province' values use Lao-French romanization
(`Attapu`, `Vientiane[prefecture]`, ...) that don't match `laos_reference.csv`'s `province_en`
column. The mapping below was constructed by matching each raw name to its `laos_reference.csv`
counterpart using the raw-province list printed in Laos's own original notebook's executed output
(`PROV_ORDER`, 17 units — Bokeo has zero NDT7 tests, matching that notebook's own note) — it has
**not been verified against the live raw parquet** since that file isn't available on this machine.
This is the single highest-risk part of this port; double-check it against real data before
trusting any Laos province-level number downstream.

**Not yet executed in this repo** — Laos raw clean parquet not available locally. Needs a
run+verify pass (e.g. by whoever has the full local dataset) before the outputs below can be
trusted, following the same handoff pattern already used for Cambodia/Thailand/Vietnam.

**Outputs:**
- `data/exports/ndt7_laos_province_quarterly.csv` — Broadband
- `data/exports/ndt7_mobile_laos_province_quarterly.csv` — Mobile/Cellular


In [1]:
import duckdb
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW_PARQUET = '../../../data/ndt7/la/mlab_la_clean.parquet'
LA_REF_CSV = '../../../data/reference/laos_reference.csv'

ZOOM = 16
N_TILES = 2 ** ZOOM
MIN_TILE_TESTS = 3

### 1. Tile-Binning + Province-Quarter Aggregation (DuckDB)

All heavy row-level work (filtering, quarter-labeling, zoom-16 mercator tile assignment, GROUP BY tile x quarter x type x network_type) happens in one DuckDB SQL query against the raw parquet — no Python-side batching.

In [2]:
con = duckdb.connect()
con.execute("SET memory_limit='3GB'")                        # PH/ID ใหญ่ ต้องตั้ง
con.execute("SET temp_directory='../../../.tmp/duckdb'")   # ที่พักตอน spill
con.execute("SET preserve_insertion_order=false")

sql = f"""
WITH filtered AS (
    SELECT
        mean_throughput_mbps,
        min_rtt,
        type, network_type, province,
        year,
        CAST(CEIL(month / 3.0) AS INT) AS qtr,
        -- zoom-16 Web Mercator tile — ใช้เป็นคอลัมน์วินิจฉัยเท่านั้น ไม่ได้ใช้คิดค่าเฉลี่ย
        CAST(LEAST(FLOOR((longitude + 180) / 360 * 65536), 65535) AS BIGINT) AS tx,
        CAST(LEAST(FLOOR((1 - (ln(tan(radians(LEAST(GREATEST(latitude, -85.05112878), 85.05112878)))
             + 1.0/cos(radians(LEAST(GREATEST(latitude, -85.05112878), 85.05112878))))) / pi()) / 2 * 65536), 65535) AS BIGINT) AS ty
    FROM read_parquet('{RAW_PARQUET}')
    WHERE mean_throughput_mbps > 0
      AND province IS NOT NULL
      AND network_type IN ('broadband', 'cellular')
)
SELECT
    province, network_type, type,
    (CAST(year AS VARCHAR) || '-Q' || CAST(qtr AS VARCHAR)) AS year_q,
    AVG(mean_throughput_mbps)                       AS avg_thr,
    AVG(CASE WHEN min_rtt < 2000 THEN min_rtt END)  AS avg_lat,
    COUNT(*)                                        AS test_count,
    COUNT(DISTINCT tx * 65536 + ty) AS n_tiles
FROM filtered
GROUP BY province, network_type, type, year_q
"""

tile_agg_all = con.execute(sql).df()
print(f"province x quarter x type x network rows: {len(tile_agg_all):,}")
print(f"quarters: {len(tile_agg_all['year_q'].unique())} | province: {tile_agg_all['province'].nunique()}")
print(tile_agg_all.groupby('network_type')['test_count'].sum().apply(lambda x: f'{x:,}'))

province x quarter x type x network rows: 372
quarters: 12 | province: 17
network_type
broadband    69,925
cellular     50,387
Name: test_count, dtype: str


### 2. Province Name Mapping — Raw (Lao romanization) → Reference (`province_en`)

Applied here, after DuckDB's tile-level aggregation — the intermediate result is small (thousands of rows, not hundreds of thousands), so this stays a plain pandas `.map()` exactly like the original.

In [3]:
# Raw NDT7 parquet 'province' values (Lao-French romanization, from the original notebook's
# executed PROV_ORDER output) -> laos_reference.csv 'province_en'.
# Verified against the real raw parquet: the map itself was correct, but was originally never
# applied to tile_agg_all (dead code) -- fixed here. All 17 raw values now match reference
# (Bokeo correctly has zero raw rows, confirmed against the real data).
PROVINCE_MAP = {
    'Vientiane[prefecture]': 'Vientiane Capital',
    'Vientiane': 'Vientiane',
    'Bolikhamxai': 'Bolikhamsai',
    'Attapu': 'Attapeu',
    'Champasak': 'Champasak',
    'Louangphrabang': 'Luang Prabang',
    'Xékong': 'Xekong',
    'Xaignabouri': 'Xaignabouli',
    'Xaisômboun': 'Xaisomboun',
    'Houaphan': 'Houaphan',
    'Khammouan': 'Khammouane',
    'Savannakhét': 'Savannakhet',
    'Phôngsali': 'Phongsaly',
    'Oudômxai': 'Oudomxay',
    'Saravan': 'Salavan',
    'Xiangkhoang': 'Xiangkhouang',
    'LouangNamtha': 'Luang Namtha',
    # Bokeo: zero NDT7 test volume per the original notebook (17/18 provinces present) — no raw
    # name to map, left out of this dict deliberately, not a missed case.
}

tile_agg_all['province'] = tile_agg_all['province'].replace(PROVINCE_MAP)
print(f"Applied PROVINCE_MAP ({len(PROVINCE_MAP)} entries)")

Applied PROVINCE_MAP (17 entries)


### 3. Province-Level Weighted Aggregation (per network type)

In [4]:
def build_province_quarterly(tile_agg_all, network_type, ref):
    d = tile_agg_all[tile_agg_all['network_type'] == network_type]
    print(f"[{network_type}] province x quarter x type rows: {len(d):,}")

    dl = d[d['type'] == 'download'].rename(columns={
        'avg_thr': 'avg_d_mbps', 'avg_lat': 'avg_lat_ms_wt', 'test_count': 'total_tests'})
    ul = d[d['type'] == 'upload'].rename(columns={'avg_thr': 'avg_u_mbps'})

    dl_stats = dl[['year_q', 'province', 'avg_d_mbps', 'avg_lat_ms_wt', 'total_tests', 'n_tiles']]
    ul_stats = ul[['year_q', 'province', 'avg_u_mbps']]

    master = pd.merge(dl_stats, ul_stats, on=['year_q', 'province'], how='outer')
    master = master.rename(columns={'year_q': 'quarter'})
    master['year'] = master['quarter'].str.slice(0, 4).astype(int)
    master['quarter.1'] = master['quarter'].str.slice(6, 7).astype(int)

    # NDT7 ใช้ total_tests อย่างเดียว ไม่ใช้ n_tiles เป็นเกณฑ์ (Ookla ยังใช้ทั้งคู่)
    # เหตุผล: NDT7 ได้พิกัดจาก MaxMind ซึ่งเป็น city centroid ทุก test ในเมืองเดียวกันจึงตกลง tile
    # เดียวกัน n_tiles จึงวัด "จังหวัดนี้มีกี่เมืองใน MaxMind" ไม่ได้วัดการกระจายตัวของข้อมูล
    # (ลาวทั้งประเทศมีพิกัดต่างกัน 33 จุด n_tiles สูงสุด = 3 -> เกณฑ์ >=5 เป็นไปไม่ได้)
    # คอลัมน์ n_tiles ยังเก็บไว้ให้ดูใน "Data Quality" ของ EDA
    master['is_reliable'] = master['total_tests'] >= 100
    print(f"[{network_type}] province x quarter rows: {len(master)} | "
          f"reliable: {master['is_reliable'].sum()} ({master['is_reliable'].mean():.1%})")

    master = master.merge(
        ref[['province_en', 'region', 'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021',
             'density_per_km2', 'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']],
        left_on='province', right_on='province_en', how='left'
    ).drop(columns=['province_en'])

    missing_ref = master[master['region'].isna()]['province'].unique()
    if len(missing_ref):
        print(f"[{network_type}] WARNING — no reference match: {list(missing_ref)}")

    return master


EXPORT_COLS = ['province', 'quarter', 'year', 'quarter.1', 'avg_d_mbps', 'avg_u_mbps',
               'avg_lat_ms_wt', 'total_tests', 'n_tiles', 'is_reliable', 'region',
               'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021', 'density_per_km2',
               'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']

In [5]:
ref = pd.read_csv(LA_REF_CSV)

---
## Part 1 — Broadband

In [6]:
broadband_master = build_province_quarterly(tile_agg_all, 'broadband', ref)
broadband_master.head()

[broadband] province x quarter x type rows: 291
[broadband] province x quarter rows: 149 | reliable: 37 (24.8%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Attapeu,4.008179,144.934000,2.0,1.0,1.511381,2023,1,False,South,4,167000,2526.05,16,8080.44,258574.08
1,2023-Q1,Bolikhamsai,14.844529,154.954143,322.0,1.0,11.080294,2023,1,True,Central,3,330700,2526.05,22,8080.44,258574.08
2,2023-Q1,Champasak,12.453606,148.090602,103.0,3.0,8.953650,2023,1,True,South,1,781200,2526.05,51,8080.44,258574.08
3,2023-Q1,Houaphan,6.562082,201.068250,16.0,1.0,1.864213,2023,1,False,North,3,317100,2526.05,19,8080.44,258574.08
4,2023-Q1,Khammouane,14.489528,72.431000,4.0,2.0,16.032535,2023,1,False,Central,2,451300,2526.05,28,8080.44,258574.08


In [7]:
out_bb = broadband_master[EXPORT_COLS].copy()
OUT_PATH_BB = '../../../data/exports/ndt7_laos_province_quarterly.csv'
out_bb.to_csv(OUT_PATH_BB, index=False)
print(f"Exported {len(out_bb)} rows -> {OUT_PATH_BB}")
out_bb.head(3)

Exported 149 rows -> ../../../data/exports/ndt7_laos_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Attapeu,2023-Q1,2023,1,4.008179,1.511381,144.934000,2.0,1.0,False,South,4,167000,2526.05,16,8080.44,258574.08
1,Bolikhamsai,2023-Q1,2023,1,14.844529,11.080294,154.954143,322.0,1.0,True,Central,3,330700,2526.05,22,8080.44,258574.08
2,Champasak,2023-Q1,2023,1,12.453606,8.953650,148.090602,103.0,3.0,True,South,1,781200,2526.05,51,8080.44,258574.08


---
## Part 2 — Mobile/Cellular

In [8]:
mobile_master = build_province_quarterly(tile_agg_all, 'cellular', ref)
mobile_master.head()

[cellular] province x quarter x type rows: 81
[cellular] province x quarter rows: 41 | reliable: 24 (58.5%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Bolikhamsai,17.378273,141.086562,169,1,6.814485,2023,1,True,Central,3,330700,2526.05,22,8080.44,258574.08
1,2023-Q1,Champasak,5.916164,182.263444,18,2,1.257469,2023,1,False,South,1,781200,2526.05,51,8080.44,258574.08
2,2023-Q1,Vientiane Capital,12.079012,177.178575,388,1,6.551746,2023,1,True,Central,1,1009300,2526.05,257,8080.44,258574.08
3,2023-Q2,Bolikhamsai,13.513299,128.069955,336,1,7.187030,2023,2,True,Central,3,330700,2526.05,22,8080.44,258574.08
4,2023-Q2,Champasak,15.303949,121.096800,5,1,5.365900,2023,2,False,South,1,781200,2526.05,51,8080.44,258574.08


In [9]:
out_mb = mobile_master[EXPORT_COLS].copy()
OUT_PATH_MB = '../../../data/exports/ndt7_mobile_laos_province_quarterly.csv'
out_mb.to_csv(OUT_PATH_MB, index=False)
print(f"Exported {len(out_mb)} rows -> {OUT_PATH_MB}")
out_mb.head(3)

Exported 41 rows -> ../../../data/exports/ndt7_mobile_laos_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Bolikhamsai,2023-Q1,2023,1,17.378273,6.814485,141.086562,169,1,True,Central,3,330700,2526.05,22,8080.44,258574.08
1,Champasak,2023-Q1,2023,1,5.916164,1.257469,182.263444,18,2,False,South,1,781200,2526.05,51,8080.44,258574.08
2,Vientiane Capital,2023-Q1,2023,1,12.079012,6.551746,177.178575,388,1,True,Central,1,1009300,2526.05,257,8080.44,258574.08


## Summary

- Input: Laos NDT7 raw test records, already province-joined + ISP-classified
- Output: province x quarter aggregates for Broadband and Mobile separately, tile-binned at
  Ookla's zoom-16 resolution, same `is_reliable` threshold as every Ookla country notebook and
  the other NDT7 "tigger" prep notebooks
- Engine: DuckDB (was: manual pyarrow-batch-streaming loop in pandas) — verified to reproduce
  the prior pandas-based export exactly (float-precision-only differences)